# ML-02 - Research Question and Provisional Lane

This notebook frames my provisional capstone lane before modeling. I am starting with the decision, action, unit of analysis, and risk of being wrong, then using a few starter-dataset numbers to check whether the lane is worth exploring.


## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring.** I want to build a ranked review queue that helps a content or SEO reviewer decide which existing pages deserve attention first. This lane fits the starter data because the dataset is already one row per pseudonymized content item, with search exposure, clicks, engagement, freshness, content depth, and trend fields. The lane also fits the business decision: when review time is limited, the useful output is not just a prediction, but a prioritized list with reason codes such as declining with demand, low CTR for a visible page, thin content, or page-one decay risk.

I may refine the exact target by Week 4, especially if the warehouse data supports a cleaner future-window outcome. For now, the project question is about prioritization: which pages should be reviewed first, and why?


In [1]:
lane = "Refresh / Content Opportunity Scoring"
unit_of_analysis = "one pseudonymized content item / page"
output = "ranked review queue with scores, suggested actions, and reason codes"

print(f"Lane: {lane}")
print(f"Unit of analysis: {unit_of_analysis}")
print(f"Output: {output}")


Lane: Refresh / Content Opportunity Scoring
Unit of analysis: one pseudonymized content item / page
Output: ranked review queue with scores, suggested actions, and reason codes


## 2. The question: decision, action, cost of a wrong call

**Search question:** Which existing content items should a reviewer inspect first for refresh, CTR review, expansion, protection, pruning, or monitoring, using only safe observed signals available before the recommendation?

**Decision improved:** A content or SEO lead has limited review capacity and needs to choose which pages enter the next refresh queue. The decision is not "is this page good or bad?" It is "should this page be reviewed before other pages?"

**Action someone could take:** The reviewer would inspect the ranked queue, read the reason codes, and choose an action such as refresh content, rewrite title or metadata, expand thin visible content, review engagement fit, protect a strong page, or monitor a page that is not urgent.

**Cost of a wrong recommendation:** A false positive wastes reviewer time and may cause unnecessary edits to a page that did not need attention. A false negative leaves a valuable declining or under-capturing page unreviewed, which could mean missed clicks, missed sessions, or delayed recovery. Because the action still needs human review, the first version should optimize for a high-quality top-K queue rather than automatic publishing decisions.

**Why data or ML can help:** A plain rule can catch obvious cases, but the priority decision mixes many signals: impressions, clicks, CTR, average position, age, freshness, content depth, engagement, and trend context. A transparent baseline should come first. ML is only worth adding if it ranks the top review candidates better than that baseline while still leaving reason codes and caveats for the human reviewer.


In [2]:
problem_frame = {
    "decision": "which pages should enter the review queue first",
    "actor": "content or SEO reviewer",
    "action": "inspect ranked candidates and choose refresh / CTR review / expansion / monitoring actions",
    "wrong_call_cost": "wasted review time, unnecessary edits, or missed declining/opportunity pages",
    "metric_to_start": "precision@K for the top review queue, compared with a transparent baseline",
}

for key, value in problem_frame.items():
    print(f"{key}: {value}")


decision: which pages should enter the review queue first
actor: content or SEO reviewer
action: inspect ranked candidates and choose refresh / CTR review / expansion / monitoring actions
wrong_call_cost: wasted review time, unnecessary edits, or missed declining/opportunity pages
metric_to_start: precision@K for the top review queue, compared with a transparent baseline


## 3. Quick look at the data (2-3 real numbers)

I am using the starter CSV only for week-one framing. These numbers are aggregated and public-safe: no client names, URLs, titles, or raw queries. The goal is to see whether there is enough volume and enough prioritization need to justify the lane.


In [3]:
from pathlib import Path
import pandas as pd

# Find the repo root whether this notebook is run from the repo root or from work/notebooks.
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data/raw/content_refresh_anonymized.csv").exists()
)

data_path = repo_root / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

summary = pd.DataFrame(
    [
        ("Rows / content items", len(df), "one row per pseudonymized page"),
        ("Pseudonymized clients", df["client_id"].nunique(), "for grouping and holdout validation, not as features"),
        ("Pages with >=500 impressions", int((df["impressions_90d"] >= 500).sum()), "enough visible inventory for a review queue"),
        ("Declining-label rows", int((df["trend_direction"] == "down").sum()), "starter proxy label, not a future outcome"),
        ("Low-CTR visible pages", int(((df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).sum()), "possible CTR review candidates"),
        ("Page-one decay-risk pages", int(((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)).sum()), "visible older pages that may deserve protection or review"),
    ],
    columns=["number", "value", "why it matters"],
)

summary


,number,value,why it matters
0,Rows / content items,30000,one row per pseudonymized page
1,Pseudonymized clients,32,"for grouping and holdout validation, not as fe..."
2,Pages with >=500 impressions,16726,enough visible inventory for a review queue
3,Declining-label rows,16262,"starter proxy label, not a future outcome"
4,Low-CTR visible pages,9759,possible CTR review candidates
5,Page-one decay-risk pages,7076,visible older pages that may deserve protectio...


The quick check supports the lane. The starter file has **30,000 content items across 32 pseudonymized clients**, so the unit of analysis is large enough for ranking work. **16,726 pages have at least 500 impressions in the trailing 90-day window**, which means there are many visible pages where a review decision could matter. The starter data also shows **9,759 visible low-CTR candidates** and **7,076 page-one decay-risk candidates**, so there are enough concrete cases for reason-coded prioritization instead of a vague "train a model" project.

I will treat the starter declining label carefully because it comes from current trend fields. It is useful for learning the workflow and comparing a baseline queue, but a stronger capstone should move toward a future-window label from the warehouse if the data contract supports it.


## 4. Careful words: what I can and can't claim

What I can claim if the evidence supports it: this project can produce an observed, decision-support ranking of pseudonymized pages that appear more worth human review than others under the chosen scoring policy. It can compare a transparent baseline against a learned ranking using top-K metrics. It can say which safe signals are associated with higher review priority in this dataset.

What I cannot claim from this data alone: I cannot claim that a refresh will cause recovery, that low CTR proves a title or meta description is bad, that the model understands Google's algorithm, or that the starter declining label is the same as a future business outcome. Any recommendation should be treated as a review prompt for a human, not an automatic instruction to edit or publish.


In [4]:
claims = pd.DataFrame(
    [
        ("Allowed", "Observed and directional patterns in safe aggregated data"),
        ("Allowed", "Decision-support ranking for human review"),
        ("Allowed", "Baseline vs model comparison using top-K queue metrics"),
        ("Not allowed", "Causal proof that editing a page caused recovery"),
        ("Not allowed", "Claims about Google algorithm factors, raw queries, URLs, titles, or client identity"),
        ("Not allowed", "Automatic publishing decisions without human review"),
    ],
    columns=["claim type", "statement"],
)

claims


,claim type,statement
0,Allowed,Observed and directional patterns in safe aggr...
1,Allowed,Decision-support ranking for human review
2,Allowed,Baseline vs model comparison using top-K queue...
3,Not allowed,Causal proof that editing a page caused recovery
4,Not allowed,"Claims about Google algorithm factors, raw que..."
5,Not allowed,Automatic publishing decisions without human r...


## Self-check

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.
